In [1]:
from build123d import *

In [2]:
class BearingDisk(Compound):
    def __init__(
        self,
        diameter: float,
        thickness: float,
        bearing_diameter: float,
        bearing_depth: float,
        shaft_diameter: float,
        parent=None
    ):
        with BuildPart() as disk_part:
            # 主体圆盘
            with BuildSketch():
                Circle(diameter / 2)
            extrude(amount=thickness)

            # 轴承座（浅孔）
            with BuildSketch(Plane.XY):
                Circle(bearing_diameter / 2)
            extrude(amount=-bearing_depth, mode=Mode.SUBTRACT)

            # 中轴孔（贯穿）
            with BuildSketch(Plane.XY):
                Circle(shaft_diameter / 2)
            extrude(amount=thickness, mode=Mode.SUBTRACT)

            # ⚡ 父 joint：固定
            RigidJoint(
                label="shaft_joint",
                joint_location=Location((0, 0, thickness / 2),(0,0,90))
            )

        super().__init__(
            disk_part.part.wrapped,
            joints=disk_part.part.joints,
            parent=parent
        )


In [3]:
class Gimbal(Compound):
    def __init__(
        self, 
        disc_diameter_gap: float,
        disc_thickness_gap: float,
        frame_thickness_diameter: float, 
        corner_radius: float,
        connector_length: float,
        connector_radius: float,
        # disk
    ):
        """
        创建一个用于支撑转盘的 Gimbal 框架
        disc_diameter_gap: 转盘直径 + 留出间隙
        disc_thickness_gap: 转盘厚度 + 留出间隙
        frame_thickness_diameter: 矩形管管径
        corner_radius: 方孔圆角半径
        connector_length: 左右连接柱长度
        connector_radius: 左右连接柱半径
        """
        a = disc_diameter_gap + frame_thickness_diameter/2
        b = disc_thickness_gap + frame_thickness_diameter/2
        r = corner_radius
        tube_radius = connector_radius
        tube_length = connector_length
        # self.disk = disk

        with BuildPart() as gimbal_part:
            # -----------------------------
            # 1️⃣ 构建圆角矩形路径
            with BuildLine() as frame:
                start = (-a/2 + r, -b/2)
                Line(start, (a/2 - r, -b/2))
                JernArc(start=(a/2 - r, -b/2), tangent=(1, 0), radius=r, arc_size=90)
                Line((a/2, -b/2 + r), (a/2, b/2 - r))
                JernArc(start=(a/2, b/2 - r), tangent=(0, 1), radius=r, arc_size=90)
                Line((a/2 - r, b/2), (-a/2 + r, b/2))
                JernArc(start=(-a/2 + r, b/2), tangent=(-1, 0), radius=r, arc_size=90)
                Line((-a/2, b/2 - r), (-a/2, -b/2 + r))
                JernArc(start=(-a/2, -b/2 + r), tangent=(0, -1), radius=r, arc_size=90)
            rectangle_wire = frame.wire()

            # -----------------------------
            # 2️⃣ sweep 圆角矩形管
            # TODO: start and tangent
            start = rectangle_wire @ 0
            tangent = rectangle_wire % 0 
            with BuildSketch(Plane(origin=start, z_dir=tangent)) as sk:
                Circle(frame_thickness_diameter/2)
            sweep(sk.sketch, rectangle_wire, is_frenet=True)

            # -----------------------------
            # 3️⃣ 左右中点圆柱（extrude）沿 X 方向
            left_mid = (-a/2, 0)
            right_mid = (a/2, 0)
        
            for mid, direction in [(left_mid, -1), (right_mid, 1)]:
                plane = Plane(origin=(mid[0], mid[1], 0), z_dir=(direction, 0, 0))
                with BuildSketch(plane) as skc:
                    Circle(connector_radius)
                extrude(skc.sketch, connector_length)
                
            # constraints 
            RevoluteJoint(
                label="bearing_axis",
                axis=Axis((0, 0, 0), (0, 1, 0)),
                angular_range=(0, 360)
            )
            
                                        
        # gimbal_part.part.joints["bearing_axis"].connect_to(disk.joints["shaft_joint"])
        super().__init__(
            gimbal_part.part.wrapped,
            joints=gimbal_part.part.joints,
            # children=[disk]
        )
        


In [6]:
import numpy as np

In [7]:
from build123d import *
import numpy as np

class Frame(Compound):
    def __init__(self, cube_size, beam,label=None):
        
        box = Box(cube_size, cube_size, cube_size)
        center = Vector(0, 0, 0)
        
        joints = []
        
        with BuildPart() as bp:
            
            # ========= 1️⃣ 生成框架 =========
            for edge in box.edges():
                
                mid = edge.position_at(0.5)
                direction = edge.tangent_at(0)
                outward = (mid - center).normalized()
                
                plane = Plane(
                    origin=mid - outward * (beam / np.sqrt(2)),
                    z_dir=direction
                )
                
                with BuildSketch(plane):
                    Rectangle(beam, beam)
                
                sweep(path=edge)
            
            # part = bp.part
        
        # ========= 2️⃣ 添加 joints =========
        # 六个面
        # face_dirs = [
        #     Vector(1,0,0), Vector(-1,0,0),
        #     Vector(0,1,0), Vector(0,-1,0),
        #     Vector(0,0,1), Vector(0,0,-1),
        # ]
        
        # for i, normal in enumerate(face_dirs):
        #     face_center = normal * (cube_size / 2)
            
        #     joint = RevoluteJoint(
        #         label=f"face_{i}",
        #         parent=bp.part,
        #         location=Location(
        #             Plane(
        #                 origin=face_center,
        #                 z_dir=normal  # 👉 旋转轴方向
        #             )
        #         )
        #     )
            
        #     joints.append(joint)
            x_faces = [Vector(cube_size/2, 0, 0), Vector(-cube_size/2, 0, 0)]
            for i, face_center in enumerate(x_faces):
                RevoluteJoint(
                    label=f"x_face_{i}",
                    axis=Axis(face_center, (0, 0, 1)),
                    angular_range=(-90, 90)
    
                )
        
        # ========= 3️⃣ 初始化 Compound =========
        super().__init__(
            bp.part.wrapped,
            joints=bp.part.joints,
            label=label
        )
        # super().__init__(part.wrapped, joint)
        
        # self.joints = joints


In [10]:
cube_size = 100
beam_size = 5
g1 = Gimbal(
    disc_diameter_gap=50+4, 
    disc_thickness_gap=5+4, 
    frame_thickness_diameter=5, 
    corner_radius=2, 
    connector_length=20, 
    connector_radius=2,
)
d1 = BearingDisk(
    diameter=50,
    thickness=5,
    bearing_diameter=22.2,
    bearing_depth=7,
    shaft_diameter=8.2,
)
g2 = Gimbal(
    disc_diameter_gap=50+4, 
    disc_thickness_gap=5+4, 
    frame_thickness_diameter=5, 
    corner_radius=2, 
    connector_length=20, 
    connector_radius=2,
)
d2 = BearingDisk(
    diameter=50,
    thickness=5,
    bearing_diameter=22.2,
    bearing_depth=7,
    shaft_diameter=8.2,
    # parent=g2
)
g1.joints["bearing_axis"].connect_to(d1.joints["shaft_joint"], angle=50)
g2.joints["bearing_axis"].connect_to(d2.joints["shaft_joint"], angle=50)
gc1 = Compound(
    label="gc1", children=[g1,d1], 
)
gc2 = Compound(
    label="gc2", children=[g2,d2], 
)
gc1.joints["gimbal_rotation"] = RigidJoint(
    label="gimbal_rotation",
    joint_location=Location((0, 0, 0),(0,90,90)),
    to_part=gc1
)
gc2.joints["gimbal_rotation"] = RigidJoint(
    label="gimbal_rotation",
    joint_location=Location((0, 0, 0),(0,90,90)),
    to_part=gc2
)
f = Frame(cube_size,beam_size,label="frame")
f.joints["x_face_0"].connect_to(gc1.joints["gimbal_rotation"], angle=90)
f.joints["x_face_1"].connect_to(gc2.joints["gimbal_rotation"], angle=0)

In [11]:
cmg = Compound(label="cmg",children=[f,gc1,gc2])

In [12]:
cmg

Compound at 0x7f6ad43a3fe0, label(cmg), #children(3)

In [27]:
g1.solids()[0]

In [19]:

cube_size = 100
beam = 5
f = Frame(cube_size, beam, label="frame")
f.joints["x_face_0"].connect_to(g1.joints["gimbal_rotation"], angle=90)
f.joints["x_face_1"].connect_to(g2.joints["gimbal_rotation"], angle=0)

In [12]:
cmg = Compound(label="cmg", children=[f, g1, g2])

In [14]:
g1

Gimbal at 0x7f68341eeb40, label(), #children(1)

In [13]:
cmg

Compound at 0x7f68341edd90, label(cmg), #children(3)